# Cảnh báo ngủ gật cho người lái xe

**Sổ Nghiên Cứu AI, dự án mẫu 5/10.** Tỉ lệ khung mắt (EAR) từ điểm mốc khuôn mặt, so sánh ngưỡng cố định, ngưỡng cá nhân hóa và kết hợp ngáp.

**Câu hỏi nghiên cứu:** Ngưỡng cá nhân hóa và kết hợp ngáp có giảm báo động giả so với ngưỡng cố định không?

**Giả thuyết:** Cách kết hợp giảm báo động giả mỗi giờ ít nhất 30% mà không bỏ sót thêm sự kiện buồn ngủ.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

**An toàn:** tuyệt đối không thử nghiệm khi đang lái xe thật. Chỉ quay video mô phỏng khi ngồi yên, có người lớn giám sát.

Phần 1 tính EAR từ video của em. Phần 2 chạy ngay với **chuỗi EAR giả lập** của nhiều người có mắt to nhỏ khác nhau.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
%pip install -q mediapipe

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)
import cv2, urllib.request
FPS = 15

## Phần 1. EAR từ video
EAR = (khoảng cách dọc mắt) / (độ rộng mắt). Mắt nhắm thì EAR giảm mạnh. Đặt video vào `video_ngu_gat.mp4`.

In [ ]:
MAT_TRAI, MAT_PHAI = [33, 160, 158, 133, 153, 144], [362, 385, 387, 263, 373, 380]
MIENG = [61, 81, 311, 291, 402, 178]
def ti_le(p, idx):
    a, b, c, d, e, f = [p[i] for i in idx]
    return (np.linalg.norm(b - f) + np.linalg.norm(c - e)) / (2 * np.linalg.norm(a - d) + 1e-6)
EAR_THAT = None
if os.path.exists("video_ngu_gat.mp4"):
    import mediapipe as mp
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", "face.task")
    det = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path="face.task"), running_mode=vision.RunningMode.VIDEO))
    cap = cv2.VideoCapture("video_ngu_gat.mp4"); fps = cap.get(cv2.CAP_PROP_FPS) or 30; e, m, i = [], [], 0
    while True:
        ok, f = cap.read()
        if not ok: break
        r = det.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(f, cv2.COLOR_BGR2RGB)), int(i * 1000 / fps)); i += 1
        if r.face_landmarks:
            p = np.array([[d.x, d.y] for d in r.face_landmarks[0]]); e.append((ti_le(p, MAT_TRAI) + ti_le(p, MAT_PHAI)) / 2); m.append(ti_le(p, MIENG))
        else: e.append(np.nan); m.append(np.nan)
    EAR_THAT, MAR_THAT, FPS = np.array(e), np.array(m), fps
    plt.figure(figsize=(11, 2.5)); plt.plot(np.arange(len(e)) / fps, e); plt.xlabel("giây"); plt.ylabel("EAR"); plt.show()
else:
    print("Chưa có video_ngu_gat.mp4. Dùng dữ liệu giả lập ở phần 2.")

## Phần 2. Mô phỏng nhiều người lái
Mỗi người có EAR mắt mở khác nhau (người mắt nhỏ có EAR thấp tự nhiên). Có chớp mắt bình thường, và các **sự kiện buồn ngủ** (nhắm mắt lâu, thường đi kèm ngáp).

In [ ]:
def mo_phong(nguoi, phut=20, rng=None):
    rng = rng or np.random.default_rng(nguoi); n = phut * 60 * FPS
    goc = rng.uniform(0.19, 0.34); ear = goc + rng.normal(0, 0.012, n); mar = 0.3 + rng.normal(0, 0.03, n); su_kien = []
    for t in np.where(rng.random(n) < 0.3 / FPS)[0]: ear[t:t + int(0.25 * FPS)] = goc * 0.35          # chớp mắt
    for t in rng.choice(np.arange(2 * 60 * FPS, n - 60 * FPS), size=rng.integers(2, 5), replace=False):
        d = int(rng.uniform(1.6, 3.5) * FPS); ear[t:t + d] = goc * rng.uniform(0.3, 0.5); su_kien.append(t)
        if rng.random() < 0.7: y = t - int(rng.uniform(20, 90) * FPS); mar[y:y + 3 * FPS] = 0.8               # ngáp trước đó
    for t in rng.choice(np.arange(n), size=rng.integers(2, 6)): mar[t:t + 2 * FPS] = 0.75                     # nói chuyện, cười
    for t in rng.choice(np.arange(n - 5 * FPS), size=rng.integers(4, 9)): ear[t:t + int(rng.uniform(1.5, 2.2) * FPS)] = goc * 0.55   # liếc xuống bảng đồng hồ
    return ear, mar, sorted(su_kien)
NGUOI = [mo_phong(i) for i in range(12)]
e, m, sk = NGUOI[0]
plt.figure(figsize=(11, 2.5)); plt.plot(np.arange(len(e)) / FPS / 60, e, lw=.5)
for t in sk: plt.axvline(t / FPS / 60, color="r", alpha=.4)
plt.xlabel("phút"); plt.ylabel("EAR"); plt.title("Người 1: vạch đỏ là sự kiện buồn ngủ thật"); plt.show()

In [ ]:
def canh_bao(ear, mar, cach):
    n = len(ear)
    if cach == "co_dinh": nguong = np.full(n, 0.21)
    else: nguong = np.full(n, 0.75 * np.nanmedian(ear[:60 * FPS]))                 # hiệu chỉnh trong 1 phút đầu
    nham = ear < nguong; dem = 0; out = []; ngap_gan = -10 ** 9
    for t in range(n):
        if mar[t] > 0.6: ngap_gan = t
        dem = dem + 1 if nham[t] else 0
        can = 1.5 if cach != "ket_hop" else (1.0 if t - ngap_gan < 120 * FPS else 2.0)
        if dem == int(can * FPS): out.append(t)
    return out
def danh_gia(cach):
    bat, tong, gia, gio, tre = 0, 0, 0, 0, []
    for ear, mar, sk in NGUOI:
        cb = canh_bao(ear, mar, cach); gio += len(ear) / FPS / 3600; dung = set()
        for s in sk:
            tong += 1; k = [c for c in cb if s <= c <= s + 4 * FPS]
            if k: bat += 1; tre.append((k[0] - s) / FPS); dung.add(k[0])
        gia += len([c for c in cb if c not in dung])
    return {"Recall sự kiện": bat / tong, "Báo động giả mỗi giờ": gia / gio, "Độ trễ TB (giây)": np.mean(tre) if tre else np.nan}
KQ = pd.DataFrame({ten: danh_gia(c) for ten, c in {"Ngưỡng cố định 0,21 (đối chứng)": "co_dinh", "Ngưỡng cá nhân hóa": "ca_nhan", "Cá nhân hóa + ngáp": "ket_hop"}.items()}).T
print(KQ.round(2))

## Việc của em
1. Quay video mô phỏng của 8 đến 10 người (có đồng ý), ghi nhãn thời điểm "buồn ngủ" theo kịch bản.
2. Thử trong điều kiện thiếu sáng, đeo kính. Báo kết quả riêng từng điều kiện.
3. Đưa thuật toán lên Raspberry Pi hoặc điện thoại cũ, đo độ trễ thật.
4. Chọn số giây nhắm mắt và hệ số 0,75 bằng **tập kiểm định**, không chỉnh trên tập kiểm tra.